# Notebook 07 — Check inference paths and prepare balanced elementary data

The lower-learning-rate trial completed, but did not improve the 50-question validation score. This notebook prepares the next controlled experiment in two parts:

1. **CPU:** create 1,680 checked elementary examples and 120 held-out-template development diagnostics. Replace 1,680 original GSM8K training rows while preserving the total 6,726 rows and all original validation/test files and three demonstrations.
2. **GPU:** compare the same 50 validation prompts through the direct base, repeated direct base, zero-initialized LoRA, and disabled-adapter paths. This investigates path-dependent generation with **200 predictions and zero optimizer updates**.

**Kaggle:** enable Internet and a T4 GPU. Attach only the **original saved Notebook 01 output**. Earlier adapters and notebook outputs are not required. One GPU is used.

Allow roughly **30–50 minutes**, depending on setup and generation lengths. This is an estimate. This notebook does not train another adapter; review the diagnostic and data sample before the matched control/data-mixture experiment.

The 120 synthetic diagnostic questions use held-out wording and per-skill number tuples. They are development checks within these six simple skill families, not evidence of broad maths ability.


## 1. Fetch the shared GitHub code


In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys
import tempfile

REPOSITORY = "https://github.com/sahan24854/Fine-Tuning-LLM-for-math.git"
PROJECT_REF = os.environ.get("MATH_PROJECT_REF", "main")
if PROJECT_REF.startswith("-"):
    raise ValueError("Use a branch, tag, or full commit SHA as MATH_PROJECT_REF")
PROJECT = Path(tempfile.mkdtemp(prefix="math-project-"))
subprocess.run(["git", "init", "--quiet", str(PROJECT)], check=True)
subprocess.run(["git", "-C", str(PROJECT), "remote", "add", "origin", REPOSITORY], check=True)
subprocess.run(["git", "-C", str(PROJECT), "fetch", "--depth", "1", "origin", PROJECT_REF], check=True)
subprocess.run(["git", "-C", str(PROJECT), "checkout", "--detach", "--quiet", "FETCH_HEAD"], check=True)
CODE_COMMIT = subprocess.check_output(["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True).strip()
print("Repository:", REPOSITORY)
print("Code commit:", CODE_COMMIT)
print("Project checkout:", PROJECT)


Repository: https://github.com/sahan24854/Fine-Tuning-LLM-for-math.git
Code commit: 6d85b556a858b77367f047d2e8a44c0bb8ccaecd
Project checkout: /tmp/math-project-udf2pohd


From https://github.com/sahan24854/Fine-Tuning-LLM-for-math
 * branch            main       -> FETCH_HEAD
 * [new branch]      main       -> origin/main


## 2. Verify the original bundle and choose a fresh output directory

Keep the original split, pinned model/tokenizer, demonstrations and scorer. Folder/ZIP discovery is supported. If more than one bundle is attached, set `MATH_BUNDLE_PATH` to the original Notebook 01 bundle.


In [2]:
import hashlib
import shutil
from datetime import datetime, timezone

INPUT_ROOT = Path(os.environ.get("MATH_BUNDLE_PATH", "/kaggle/input"))
BUNDLE = Path(subprocess.check_output([sys.executable, str(PROJECT / "scripts/find_bundle.py"), str(INPUT_ROOT)], text=True).strip())
_ = subprocess.run([sys.executable, str(PROJECT / "scripts/verify_bundle.py"), str(BUNDLE)], check=True)
BUNDLE_HASH = hashlib.sha256((BUNDLE / "manifest.json").read_bytes()).hexdigest()
STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_ROOT = Path("/kaggle/working") / f"balanced-preparation-{BUNDLE_HASH[:8]}-{CODE_COMMIT[:8]}-{STAMP}"
DATA_OUTPUT, PATH_OUTPUT = RUN_ROOT / "data", RUN_ROOT / "inference_paths"
print("Original bundle:", BUNDLE)
print("Output:", RUN_ROOT)
print("Data settings:", (PROJECT / "configs/balanced_data.json").read_text())


Verified 17 files; code commit 6056c2f6cd6e096faf544ff026ba230f32639f9a
Original bundle: /kaggle/input/notebooks/sahan24854/01-data-and-protocol/data-protocol-20261019T082909Z-1cb02106
Output: /kaggle/working/balanced-preparation-b9d3419a-6d85b556-20261022T090202843537Z
Data settings: {
  "experiment_id": "elementary-balanced-mixture-v1",
  "seed": 1729,
  "synthetic_train_rows": 1680,
  "diagnostic_rows_per_skill": 20,
  "skills": ["add", "subtract", "equal_groups", "divide", "two_step_total", "two_step_remaining"],
  "mixture_policy": "Replace 1680 original training rows; retain all fixed demonstrations. Preserve total training row count.",
  "budget_policy": "Same epochs, effective batch size and optimizer-update count. Supervised token exposure is measured, not matched.",
  "evaluation_policy": "Original validation/test files and demonstrations stay byte-identical. New synthetic diagnostic uses held-out wording and numerical tuples; it is development data, not final test evidence."

## 3. Prepare the isolated environment

The pinned training profile supplies Transformers, PyTorch and PEFT for the zero-adapter diagnostic. It does not replace Kaggle kernel packages. Missing-`wrapt` startup warnings are separate from an actual command failure.


In [3]:
ENVIRONMENT = Path(tempfile.mkdtemp(prefix="math-preparation-env-"))
_ = subprocess.run([sys.executable, str(PROJECT / "scripts/setup_environment.py"),
    "--environment", str(ENVIRONMENT), "--profile", "training"], check=True)
PYTHON = ENVIRONMENT / "bin" / "python"
RUN_ENV = {**os.environ, "CUDA_VISIBLE_DEVICES": "0"}


Creating isolated environment with Python 3.13.15


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


Processing /tmp/math-pip-bootstrap-if409vva/pip-25.3-py3-none-any.whl


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 105.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 19.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 126.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 149.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 146.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.6/801.6 kB 31.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 30.9 MB/s  0:00:00



Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 195.3 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 36.4 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 151.7 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.9/663.9 MB 67.0 MB/s  0:00:04
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 86.8 MB/s  0:00:03
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 117.9 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 157.7 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 141.6 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 MB 129.8 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 MB 132.6 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 905.2/905.2 MB 15.9 MB/s  0:00:17
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.7/155.7 MB 135

Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


No broken requirements found.
Environment ready: /tmp/math-preparation-env-qrmdvvm7/bin/python


## 4. Run the local correctness checks

These tests check every generated example, balanced counts, independent arithmetic results, held-out templates/tuples, overlap filtering, unchanged original evaluation files and demonstrations, refusal of truncation, and the real tiny-model inference paths. No remote model weights are needed for these tests.


In [4]:
for test_file in ("test_balanced_data.py", "test_inference_paths.py"):
    _ = subprocess.run([str(PYTHON), "-m", "unittest", "discover", "-s", "tests", "-p", test_file, "-b", "-v"],
        cwd=PROJECT, check=True, env=RUN_ENV)


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'
test_complete_dataset_is_balanced_correct_and_held_out_by_tuple_and_wording (test_balanced_data.BalancedTests.test_complete_dataset_is_balanced_correct_and_held_out_by_tuple_and_wording) ... ok
test_no_silent_sequence_truncation (test_balanced_data.BalancedTests.test_no_silent_sequence_truncation) ... ok
test_variant_keeps_count_demonstrations_and_every_evaluation_byte (test_balanced_data.BalancedTests.test_variant_keeps_count_demonstrations_and_every_evaluation_byte) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.123s

OK
Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'
test_differences_align_keys_and_check_stop_boundaries (test_inference_paths.PathTests.test_differences_align_keys_and_check_stop_boundaries) ... ok
test_real_zero_adapter_probe_performs_no_training_and_saves_complete_r

## 5. Prepare and inspect the data variant

Train on 5,046 original GSM8K rows plus 1,680 synthetic rows, 280 in each skill: addition, subtraction, equal groups, exact division, multiplication then addition, and multiplication then subtraction. Retain all three original demonstrations. The 25% replacement is a declared first hypothesis, not an optimized ratio.

Solutions are short, machine-checked integer equations. This changes both skill coverage and target-solution style. The same row count preserves the planned optimizer-update count under the old schedule, but shorter responses change supervised-token exposure. The summary reports both token totals; do not claim an identical token budget.

The original validation/test files, tokenizer and demonstrations stay fixed. The new bundle gets a different identity. Keep it separate from Notebook 01; old comparison scripts intentionally reject changed bundle identities.


In [5]:
_ = subprocess.run([str(PYTHON), str(PROJECT / "scripts/prepare_balanced_data.py"),
    "--bundle", str(BUNDLE), "--output", str(DATA_OUTPUT)], check=True, cwd=PROJECT, env=RUN_ENV)
DATA_SUMMARY = json.loads((DATA_OUTPUT / "summary.json").read_text())
print(json.dumps(DATA_SUMMARY, indent=2))
print("Training sample: one example of each skill/wording template")
for line in (DATA_OUTPUT / "review_sample.jsonl").read_text().split("\n"):
    if line:
        row = json.loads(line)
        print("\n", row["template_id"], row["question"])
        print(row["solution"], "Final answer:", row["answer"], sep="\n")


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


{
  "status": "balanced_data_prepared_not_trained",
  "parent_bundle_sha256": "b9d3419aa71419626d274843f88dc6190f5828877b20a1cc67e5249e760b5c71",
  "variant_bundle_sha256": "28c1f9a0ef9865a3b389a3471a76978950db074756b510196e4c14320b9ca9cf",
  "original_rows": 6726,
  "mixed_rows": 6726,
  "retained_gsm8k": 5046,
  "synthetic_rows": 1680,
  "synthetic_per_skill": {
    "add": 280,
    "subtract": 280,
    "equal_groups": 280,
    "divide": 280,
    "two_step_total": 280,
    "two_step_remaining": 280
  },
  "diagnostic_per_skill": {
    "add": 20,
    "subtract": 20,
    "equal_groups": 20,
    "divide": 20,
    "two_step_total": 20,
    "two_step_remaining": 20
  },
  "original_supervised_tokens_per_epoch": 645749,
  "mixed_supervised_tokens_per_epoch": 525534,
  "same_row_count": true,
  "same_supervised_token_budget": false,
  "unchanged_evaluation_files": [
    "validation.jsonl",
    "gsm8k_test.jsonl",
    "svamp_test.jsonl",
    "arithmetic_test.jsonl",
    "demonstrations.json",

## 6. Check generation through four inference paths

Use identical prompts, order, batches of four, greedy settings, 512-token cap, stopping, tokenizer and loaded base weights. First repeat the direct-base path. Then attach an untrained zero-initialized LoRA adapter, and finally disable that adapter. No parameter update takes place.

The probe saves all raw completions and identifies differences in text, stop reason or generated length. A mismatch can expose a path dependence; this probe alone cannot prove its numerical cause or reproduce every historical Trainer preparation step. Even matching accuracy does not mean matching answers.


In [6]:
if not shutil.which("nvidia-smi"):
    raise RuntimeError("Enable a T4 GPU in Kaggle settings for this diagnostic.")
_ = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], check=True)
_ = subprocess.run([str(PYTHON), str(PROJECT / "scripts/check_inference_paths.py"),
    "--bundle", str(BUNDLE), "--output", str(PATH_OUTPUT)], check=True, cwd=PROJECT, env=RUN_ENV)
PATH_SUMMARY = json.loads((PATH_OUTPUT / "summary.json").read_text())
print(json.dumps(PATH_SUMMARY, indent=2))
if not PATH_SUMMARY["reproducibility_gate_passed"]:
    print("REVIEW REQUIRED: inference paths differ. Preserve the outputs and investigate before training.")


Tesla T4, 15360 MiB, 580.178.04
Tesla T4, 15360 MiB, 580.178.04


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


Loading HuggingFaceTB/SmolLM2-360M at f8027fd0eaeea54caa13c31d31b9fdc459c38b49 on Tesla T4


`torch_dtype` is deprecated! Use `dtype` instead!


Model loaded in 3.1s
direct_base: 5/50 correct
direct_repeat: 5/50 correct
zero_adapter: 5/50 correct
disabled_adapter: 5/50 correct
{
  "status": "inference_path_diagnostic_complete",
  "scope": "validation_only_no_training",
  "questions": 50,
  "predictions": 200,
  "optimizer_updates": 0,
  "groups": {
    "direct_base": {
      "count": 50,
      "expected": 50,
      "correct": 5,
      "accuracy": 0.1,
      "accuracy_wilson_95": [
        0.04347576493189041,
        0.21360231437479654
      ],
      "parseable": 31,
      "parse_rate": 0.62,
      "strict_numeric_format_rate": 0.46,
      "invalid": 19,
      "cap_hits": 9,
      "stop_reasons": {
        "final_answer_line": 40,
        "next_question": 1,
        "length": 9
      },
      "generated_tokens": 7371,
      "mean_generated_tokens": 147.42,
      "generation_seconds": 219.41613023199534
    },
    "direct_repeat": {
      "count": 50,
      "expected": 50,
      "correct": 5,
      "accuracy": 0.1,
      "accur

## 7. Verify and save the complete output

Retain both `data/` and `inference_paths/`, including their manifests. Share `inference_paths/summary.json` and `data/summary.json`, or push the executed notebook. If paths differ, also share the four prediction files. Training is a separate next step after review.

The next candidate/control comparison should use the same inference path and fresh adapters, fixed `2e-4` learning rate, three epochs, effective batch 32 and loss-based checkpoint selection. Returning to `2e-4` is an explicit reference configuration, not proof that it is optimal. The old adapters remain historical evidence; they do not replace a matched control for strong causal claims.


In [7]:
for output in (DATA_OUTPUT, PATH_OUTPUT):
    manifest = json.loads((output / "result_manifest.json").read_text())
    for name, expected in manifest["files_sha256"].items():
        if hashlib.sha256((output / name).read_bytes()).hexdigest() != expected:
            raise RuntimeError(f"Checksum mismatch: {output / name}")
ARCHIVE = Path(shutil.make_archive(str(RUN_ROOT), "zip", root_dir=RUN_ROOT))
print("Balanced data bundle:", DATA_OUTPUT / "bundle")
print("Data summary:", DATA_OUTPUT / "summary.json")
print("Inference diagnostic summary:", PATH_OUTPUT / "summary.json")
print("Complete ZIP:", ARCHIVE)
print("Save Version with outputs. Review before training the data-mixture candidate.")


Balanced data bundle: /kaggle/working/balanced-preparation-b9d3419a-6d85b556-20261022T090202843537Z/data/bundle
Data summary: /kaggle/working/balanced-preparation-b9d3419a-6d85b556-20261022T090202843537Z/data/summary.json
Inference diagnostic summary: /kaggle/working/balanced-preparation-b9d3419a-6d85b556-20261022T090202843537Z/inference_paths/summary.json
Complete ZIP: /kaggle/working/balanced-preparation-b9d3419a-6d85b556-20261022T090202843537Z.zip
Save Version with outputs. Review before training the data-mixture candidate.
